In [ ]:
# Snow Mask blend for BDD100K
import os
import random
import numpy as np
from PIL import Image

random.seed(1210)

def screen_blend(image, layer):
    '''
    input:
        image - numpy array of RGB image
        layer - numpy array of layer to blend
    '''
    result = 255.0 * (1 - (1-image/255.0) * (1-layer[:, :, None]/255.0))
    return result.astype(np.uint8)

def process_images_with_snow(input_folder, exclude_file, n_samples, output_folder, snow_masks_folder):
    # Read excluded images
    excluded_images = set()
    if exclude_file:
        with open(exclude_file, 'r') as f:
            excluded_images = set(line.strip() for line in f)

    # Get list of eligible image files
    image_files = [f for f in os.listdir(input_folder) if f.endswith('.jpg') and f not in excluded_images]

    # Randomly sample n images if there are more than n eligible images
    if len(image_files) > n_samples:
        image_files = random.sample(image_files, n_samples)

    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Create mapping file and basename file
    mapping_file = os.path.join(output_folder, 'image_mapping.txt')
    basename_file = os.path.join(output_folder, 'processed_basenames.txt')
    
    with open(mapping_file, 'w') as map_file, open(basename_file, 'w') as base_file:
        for filename in image_files:
            # Load original image
            original_image_path = os.path.join(input_folder, filename)
            original_image = Image.open(original_image_path)
            width, height = original_image.size

            # Select and resize random snow mask
            snow_mask_filename = random.choice(os.listdir(snow_masks_folder))
            snow_mask_path = os.path.join(snow_masks_folder, snow_mask_filename)
            snow_mask = Image.open(snow_mask_path)
            snow_mask_resized = snow_mask.resize((width, height)).convert("L")

            # Apply snow effect
            result_image = screen_blend(np.array(original_image), np.array(snow_mask_resized))
            result_image = Image.fromarray(result_image)

            # Generate new filename
            base_name, ext = os.path.splitext(filename)
            new_filename = f"{base_name}_snow_m{ext}"
            output_path = os.path.join(output_folder, new_filename)

            # Save processed image
            result_image.save(output_path)

            # Write mapping to file
            map_file.write(f"{filename} {new_filename}\n")

            # Write original filename to basename file
            base_file.write(f"{filename}\n")

    print(f"Processed {len(image_files)} images.")
    print(f"Mapping saved to {mapping_file}")
    print(f"Original basenames saved to {basename_file}")

# Usage example
# TODO: Modify path as needed by you
input_folder = "SnowBDD100K/base_img/train"
exclude_file = "SnowBDD100K/train_current_basename.txt"  # Set to None if not needed
n_samples = 300  # Number of images to process
output_folder = "dataset/SnowBDD100K/processed_img/train"
snow_masks_folder = "dataset/Snow100K/Snow100K-M/mask"

process_images_with_snow(input_folder, exclude_file, n_samples, output_folder, snow_masks_folder)